In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from pyproj import Transformer

ref_path = Path("/media/b085164/Elements/CALIB_26_02_25/ODyN_calib/AIRINS/base/out/ODyN_CALIB.out")

SBET_DTYPE = np.dtype([
    ("time",    np.float64), ("lat",  np.float64), ("lon",   np.float64),
    ("alt",     np.float64), ("vx",   np.float64), ("vy",    np.float64),
    ("vz",      np.float64), ("roll", np.float64), ("pitch", np.float64),
    ("heading", np.float64), ("wander",np.float64),("ax",    np.float64),
    ("ay",      np.float64), ("az",   np.float64), ("wx",    np.float64),
    ("wy",      np.float64), ("wz",   np.float64),
])

df = pd.DataFrame(np.fromfile(ref_path, dtype=SBET_DTYPE))
transformer = Transformer.from_crs("EPSG:4326", "EPSG:2056", always_xy=True)
x, y = transformer.transform(np.degrees(df["lon"].values), np.degrees(df["lat"].values))
df["x"], df["y"] = x, y

outages = {
    1: (305120, 305700),
    2: (305645, 306120),
    3: (306290, 306645),
}

exclude_zones = {
    3: [(306405, 306560)],
}

outage_distances = {}
for oid, (t0, t1) in outages.items():
    seg = df[(df["time"] >= t0) & (df["time"] <= t1)].copy()
    for (ex0, ex1) in exclude_zones.get(oid, []):
        seg = seg[(seg["time"] < ex0) | (seg["time"] > ex1)]
    dist = np.sum(np.sqrt(np.diff(seg["x"].values)**2 + np.diff(seg["y"].values)**2))
    outage_distances[oid] = dist
    print(f"Outage {oid} ({t0}–{t1} s, excl={exclude_zones.get(oid,[])}) : {len(seg)} samples,  d = {dist:.1f} m")

Outage 1 (305120–305700 s, excl=[]) : 11600 samples,  d = 7856.8 m
Outage 2 (305645–306120 s, excl=[]) : 9500 samples,  d = 3562.3 m
Outage 3 (306290–306645 s, excl=[(306405, 306560)]) : 4000 samples,  d = 2208.3 m


In [6]:
D = {1: 7856.8, 2: 3562.3, 3: 2208.3}

results = {
    1: {
        "AIRINS": {
            "INS only": dict(rmse=1.51,   q50=1.15,   q95=2.50,   std=0.88,   rmse_d=1.51/D[1]*100,   gain=None),
            "S2S":      dict(rmse=1.33,   q50=1.01,   q95=2.11,   std=0.78,   rmse_d=1.33/D[1]*100,   gain=1.51/1.33),
            "F2B":      dict(rmse=0.48,   q50=0.18,   q95=1.06,   std=0.34,   rmse_d=0.48/D[1]*100,   gain=1.51/0.48),
            "Combined": dict(rmse=0.39,   q50=0.22,   q95=0.83,   std=0.23,   rmse_d=0.39/D[1]*100,   gain=1.51/0.39),
        },
        "APX15": {
            "INS only": dict(rmse=208.65, q50=114.96, q95=390.64, std=140.59, rmse_d=208.65/D[1]*100, gain=None),
            "F2B":      dict(rmse=36.54,  q50=9.18,   q95=78.49,  std=27.59,  rmse_d=36.54/D[1]*100,  gain=208.65/36.54),
            "Combined": dict(rmse=9.39,   q50=8.05,   q95=18.05,  std=5.33,   rmse_d=9.39/D[1]*100,   gain=208.65/9.39),
        },
    },
    2: {
        "AIRINS": {
            "INS only": dict(rmse=0.97,  q50=0.46,  q95=1.90,   std=0.67,  rmse_d=0.97/D[2]*100,  gain=None),
            "S2S":      dict(rmse=0.11,  q50=0.06,  q95=0.24,   std=0.07,  rmse_d=0.11/D[2]*100,  gain=0.97/0.11),
            "F2B":      dict(rmse=0.27,  q50=0.30,  q95=0.37,   std=0.11,  rmse_d=0.27/D[2]*100,  gain=0.97/0.27),
            "Combined": dict(rmse=0.10,  q50=0.10,  q95=0.16,   std=0.04,  rmse_d=0.10/D[2]*100,  gain=0.97/0.10),
        },
        "APX15": {
            "INS only": dict(rmse=74.27,  q50=61.99, q95=120.65, std=45.19, rmse_d=74.27/D[2]*100,  gain=None),
            "F2B":      dict(rmse=10.06,  q50=5.75,  q95=23.13,  std=6.65,  rmse_d=10.06/D[2]*100,  gain=74.27/10.06),
            "Combined": dict(rmse=3.69,   q50=1.18,  q95=8.50,   std=2.71,  rmse_d=3.69/D[2]*100,   gain=74.27/3.69),
        },
    },
    3: {
        "AIRINS": {
            "INS only": dict(rmse=0.43,  q50=0.33,  q95=0.53,   std=0.25,  rmse_d=0.43/D[3]*100,  gain=None),
            "S2S":      dict(rmse=0.19,  q50=0.20,  q95=0.27,   std=0.09,  rmse_d=0.19/D[3]*100,  gain=0.43/0.19),
            "F2B":      dict(rmse=0.11,  q50=0.11,  q95=0.17,   std=0.05,  rmse_d=0.11/D[3]*100,  gain=0.43/0.11),
            "Combined": dict(rmse=0.06,  q50=0.05,  q95=0.10,   std=0.03,  rmse_d=0.06/D[3]*100,  gain=0.43/0.06),
        },
        "APX15": {
            "INS only": dict(rmse=14.96, q50=3.17,  q95=37.49,  std=12.20, rmse_d=14.96/D[3]*100, gain=None),
            "F2B":      dict(rmse=1.74,  q50=0.70,  q95=4.37,   std=1.32,  rmse_d=1.74/D[3]*100,  gain=14.96/1.74),
            "Combined": dict(rmse=0.33,  q50=0.19,  q95=0.73,   std=0.22,  rmse_d=0.33/D[3]*100,  gain=14.96/0.33),
        },
    },
}

# ----------------------------------------------------------------
# CALCUL
# ----------------------------------------------------------------
print(f"{'Outage':<10} {'IMU':<8} {'Method':<12} "
      f"{'RMSE':>7} {'Q50':>7} {'Q95':>7} {'STD':>7} "
      f"{'Drift%RMSE':>11} {'Drift%Q95':>10} {'Improv':>8}")
print("-" * 96)

for oid, imus in results.items():
    d = outage_distances[oid]
    for imu, methods in imus.items():
        ins_rmse = methods["INS only"]["rmse"]
        for method, m in methods.items():
            if m is None or any(v is None for v in m.values()):
                continue
            drift_rmse = m["rmse"] / d * 100 if d > 0 else float("nan")
            drift_q95  = m["q95"]  / d * 100 if d > 0 else float("nan")
            improv     = ins_rmse / m["rmse"] if m["rmse"] > 0 else float("nan")
            print(f"{oid:<10} {imu:<8} {method:<12} "
                  f"{m['rmse']:>7.2f} {m['q50']:>7.2f} {m['q95']:>7.2f} {m['std']:>7.2f} "
                  f"{drift_rmse:>10.3f}% {drift_q95:>9.3f}% {improv:>7.1f}x")
        print("-" * 96)

Outage     IMU      Method          RMSE     Q50     Q95     STD  Drift%RMSE  Drift%Q95   Improv
------------------------------------------------------------------------------------------------
1          AIRINS   S2S             1.33    1.01    2.11    0.78      0.017%     0.027%     1.1x
1          AIRINS   F2B             0.48    0.18    1.06    0.34      0.006%     0.013%     3.1x
1          AIRINS   Combined        0.39    0.22    0.83    0.23      0.005%     0.011%     3.9x
------------------------------------------------------------------------------------------------
1          APX15    F2B            36.54    9.18   78.49   27.59      0.465%     0.999%     5.7x
1          APX15    Combined        9.39    8.05   18.05    5.33      0.120%     0.230%    22.2x
------------------------------------------------------------------------------------------------
2          AIRINS   S2S             0.11    0.06    0.24    0.07      0.003%     0.007%     8.8x
2          AIRINS   F2B       